In [ ]:
import random, math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PAD_IDX = 0

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, d, src_vocab_size):
        super().__init__()
        self.d = d
        self.src_vocab_size = src_vocab_size
        self.emb = nn.Embedding(src_vocab_size, d, padding_idx=PAD_IDX)
        # Scale weight initialization by 1/sqrt(d)
        self.wxh = nn.Parameter(torch.randn(d, d) / math.sqrt(d))
        self.whh = nn.Parameter(torch.randn(d, d) / math.sqrt(d))
        self.b = nn.Parameter(torch.zeros(d))
        self.activation = nn.Tanh()
    def forward(self, x, mask, h=None, return_history=False):
        """ 
        x => (b, s,) => (b, s, d)
        h => (b, d)
        mask => (b, s,)
        """
        # print(f"Encoder forward: x->{x.shape}, mask->{mask.shape}")
        x = self.emb(x)
        b, s, d = x.shape
        history_list = []
        if h is None:
            # h = torch.zeros(b, d)
            h = torch.zeros(b, d, device=x.device, dtype=x.dtype) # device is crucial
        history = h
        for t in range(s):
            h_1 = self.activation(x[:,t,:] @ self.wxh + h @ self.whh + self.b)
            m_t = mask[:, t, None] # (b, 1)
            h = m_t * h_1 + (1 - m_t) * h
            # h = mask[:, t] * h_1 + (1 - mask[:, t]) * h
            if return_history:
                history_list.append(h)
        if return_history:
            history = torch.stack(history_list, dim=1)
            return h, history
        else:
            return h
class DecoderRNN(nn.Module):
    def __init__(self, encoder, tgt_vocab_size):
        super().__init__()
        self.encoder = encoder
        d = encoder.d
        self.emb = nn.Embedding(tgt_vocab_size, d, padding_idx=PAD_IDX)
        self.wxh = nn.Parameter(torch.randn(d, d))
        self.whh = nn.Parameter(torch.randn(d, d))
        self.activation = nn.Tanh()
        self.b = nn.Parameter(torch.randn(d))
        self.proj = nn.Linear(d,tgt_vocab_size, bias=False)
        self.proj.weight = self.emb.weight
    def forward(self, tgt, tgt_mask, src, src_mask, tf=0.5):
        """ 
        tgt => (b, s,)
        h => (b, d)
        mask => (b, d,)
        """
        # print(f"Decoder forward: tgt-> {tgt.shape}, tgt_mask-> {tgt_mask.shape}, src-> {src.shape}, src_mask: {src_mask.shape}")
        tgt = self.emb(tgt)
        b, s, d = tgt.shape
        h = self.encoder(src, src_mask)
        token = tgt[:, 0, :]
        outputs = []
        for t in range(1, s):
            h_1 = self.activation(token @ self.wxh + h @ self.whh + self.b)
            m_t = tgt_mask[:, t, None]
            h = m_t * h_1 + (1 - m_t) * h
            output = self.proj(h) # b, d
            outputs.append(output)
            if random.random() < tf:
                token = tgt[:, t, :]
            else:
                preds = output.argmax(dim=1)
                token = self.emb(preds)
        return torch.stack(outputs, dim=1).to(tgt.device) # b, s-1, d

In [ ]:
from tqdm import tqdm
def train_epoch(model, loader, optimizer, clip_grad=False):
    model.train()
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    grad_norm_hist = []
    for src, tgt, label, src_mask, tgt_mask in tqdm(loader, desc="calculating train loss"):
        src = src.to(device)
        tgt = tgt.to(device)
        src_mask = src_mask.to(device)
        tgt_mask = tgt_mask.to(device)
        output = model(tgt, tgt_mask, src, src_mask)
        # output = output[:, :-1]
        true_label = tgt[:, 1:]
        output = output.reshape(-1, output.shape[-1])
        true_label = true_label.reshape(-1)
        # print(output.shape, true_label.shape)
        # CrossEntropyLoss expects input is [N, C], target is [N]
        loss = criterion(output, true_label.long())
        total_loss += loss.item()
        loss.backward()
        if clip_grad:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        grad_norm_hist.append(model.encoder.whh.grad.norm())
        optimizer.step()
        optimizer.zero_grad()
    total_loss /= max(1, len(loader))
    return total_loss, torch.stack(grad_norm_hist)

In [ ]:
encoder = EncoderRNN(d=64, src_vocab_size=len(src_vocab))
encoder.whh = nn.Parameter(torch.empty(encoder.d, encoder.d))
# Apply orthogonal initialization
nn.init.orthogonal_(encoder.whh)
decoder = DecoderRNN(encoder, len(tgt_vocab))
optimizer = torch.optim.Adam(decoder.parameters(), lr=0.001)
NUM_EPOCHS = 30
# encoder.to(device)
decoder.to(device)
grad_norm_hist = []
for epoch in range(NUM_EPOCHS):
    loss, g_norm_hist = train_epoch(decoder, train_loader, optimizer, clip_grad=True)
    print(f"Epoch: {epoch+1}, loss: {loss:.4f}")
    # print(grad_norm_hist)
    grad_norm_hist.extend(g_norm_hist.cpu())
grad_norm_hist = np.array(grad_norm_hist)
plt.plot(grad_norm_hist, marker='o')
plt.yscale("log")
plt.xlabel("iteration")
plt.ylabel("grad norm (log scale)")
plt.show()

In [ ]:
class CustomLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        # One matrix for all 4 gates:
        # [input_gate, forget_gate, cell_gate, output_gate]
        self.Wx = nn.Parameter(
            torch.randn(input_size, 4 * hidden_size) / math.sqrt(input_size)
        )
        self.Wh = nn.Parameter(
            torch.randn(hidden_size, 4 * hidden_size) / math.sqrt(hidden_size)
        )
        self.b = nn.Parameter(torch.zeros(4 * hidden_size))
        # Orthogonal initialization for recurrent matrix
        nn.init.orthogonal_(self.Wh)

    def forward(self, x, state):
        h, c = state
        gates = x @ self.Wx + h @ self.Wh + self.b
        i, f, g, o = gates.chunk(4, dim=-1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)
        c_new = f * c + i * g
        h_new = o * torch.tanh(c_new)
        return h_new, c_new

class EncoderLSTM(nn.Module):
    def __init__(self, d, src_vocab_size):
        super().__init__()
        self.d = d
        self.emb = nn.Embedding(
            src_vocab_size,
            d,
            padding_idx=PAD_IDX
        )
        self.cell = CustomLSTMCell(d, d)

    def forward(self, x, mask, state=None, return_history=False):
        # x: (B, S)
        x = self.emb(x)
        B, S, D = x.shape
        if state is None:
            h = torch.zeros(B, D, device=x.device, dtype=x.dtype)
            c = torch.zeros(B, D, device=x.device, dtype=x.dtype)
        else:
            h, c = state
        history = []
        for t in range(S):
            h_new, c_new = self.cell(
                x[:, t, :],
                (h, c)
            )
            m = mask[:, t].unsqueeze(1).to(x.dtype)
            h = m * h_new + (1 - m) * h
            c = m * c_new + (1 - m) * c
            if return_history:
                history.append(h)
        if return_history:
            history = torch.stack(history, dim=1)  # (B, S, D)
            return (h, c), history
        return h, c

class DecoderLSTM(nn.Module):
    def __init__(self, encoder, tgt_vocab_size):
        super().__init__()
        self.encoder = encoder
        d = encoder.d
        self.emb = nn.Embedding(
            tgt_vocab_size,
            d,
            padding_idx=PAD_IDX
        )
        self.cell = CustomLSTMCell(d, d)
        self.proj = nn.Linear(
            d,
            tgt_vocab_size,
            bias=False
        )
        # weight tying
        self.proj.weight = self.emb.weight

    def forward(self, tgt, tgt_mask, src, src_mask, tf=0.5, do_sample=False, temperature=1.0):
        """
        tgt      : (B, T)
        tgt_mask : (B, T)
        src      : (B, S)
        src_mask : (B, S)

        returns:
        logits   : (B, T-1, vocab)
        """
        tgt_emb = self.emb(tgt)
        B, T, D = tgt_emb.shape
        h, c = self.encoder(src, src_mask)
        token = tgt_emb[:, 0, :]
        outputs = []
        for t in range(1, T):
            h_new, c_new = self.cell(
                token,
                (h, c)
            )
            m = tgt_mask[:, t].unsqueeze(1).to(tgt_emb.dtype)
            h = m * h_new + (1 - m) * h
            c = m * c_new + (1 - m) * c
            logits = self.proj(h)
            outputs.append(logits)
            # Teacher forcing
            if random.random() < tf:
                token = tgt_emb[:, t, :]
            else:
                if do_sample:
                    probs = nn.functional.softmax(logits / temperature, dim=-1)
                    probs_2d = probs.view(-1, probs.size(-1))
                    pred = torch.multinomial(probs_2d, num_samples=1).view(logits.shape[:-1])
                else:
                    pred = logits.argmax(dim=-1)
                token = self.emb(pred)
        return torch.stack(outputs, dim=1)

encoder = EncoderLSTM(d=64, src_vocab_size=len(src_vocab))
decoder = DecoderLSTM(encoder, tgt_vocab_size=len(tgt_vocab))
decoder.to(device)
optimizer = torch.optim.Adam(
    decoder.parameters(),
    lr=0.001
)

In [ ]:
class CustomGRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        # update, reset, candidate
        self.Wx = nn.Parameter(
            torch.randn(input_size, 3 * hidden_size)
            / math.sqrt(input_size)
        )
        self.Wh = nn.Parameter(
            torch.randn(hidden_size, 3 * hidden_size)
            / math.sqrt(hidden_size)
        )
        self.b = nn.Parameter(
            torch.zeros(3 * hidden_size)
        )
        nn.init.orthogonal_(self.Wh)

    def forward(self, x, h):
        x_part = x @ self.Wx
        h_part = h @ self.Wh
        x_z, x_r, x_n = x_part.chunk(3, dim=-1)
        h_z, h_r, h_n = h_part.chunk(3, dim=-1)
        z = torch.sigmoid(x_z + h_z + self.b[:self.hidden_size])
        r = torch.sigmoid(
            x_r + h_r + self.b[self.hidden_size:2*self.hidden_size]
        )
        n = torch.tanh(
            x_n
            + r * h_n
            + self.b[2*self.hidden_size:]
        )
        h_new = (1 - z) * h + z * n
        return h_new

class EncoderGRU(nn.Module):
    def __init__(self, d, src_vocab_size):
        super().__init__()
        self.d = d
        self.emb = nn.Embedding(
            src_vocab_size,
            d,
            padding_idx=PAD_IDX
        )
        self.cell = CustomGRUCell(d, d)

    def forward(
        self,
        x,
        mask,
        h=None,
        return_history=False
    ):
        x = self.emb(x)
        B, S, D = x.shape
        if h is None:
            h = torch.zeros(
                B,
                D,
                device=x.device,
                dtype=x.dtype
            )
        history = []
        for t in range(S):
            h_new = self.cell(
                x[:, t, :],
                h
            )
            m = mask[:, t].unsqueeze(1).to(x.dtype)
            h = m * h_new + (1 - m) * h
            if return_history:
                history.append(h)
        if return_history:
            history = torch.stack(
                history,
                dim=1
            )
            return h, history
        return h

class DecoderGRU(nn.Module):
    def __init__(self, encoder, tgt_vocab_size):
        super().__init__()
        self.encoder = encoder
        d = encoder.d
        self.emb = nn.Embedding(
            tgt_vocab_size,
            d,
            padding_idx=PAD_IDX
        )
        self.cell = CustomGRUCell(d, d)
        self.proj = nn.Linear(
            d,
            tgt_vocab_size,
            bias=False
        )
        self.proj.weight = self.emb.weight

    def forward(self, tgt, tgt_mask, src, src_mask, tf=0.5):
        tgt_emb = self.emb(tgt)
        B, T, D = tgt_emb.shape
        h = self.encoder(src, src_mask)
        token = tgt_emb[:, 0, :]
        outputs = []
        for t in range(1, T):
            h_new = self.cell(
                token,
                h
            )
            m = tgt_mask[:, t].unsqueeze(1).to(tgt_emb.dtype)
            h = m * h_new + (1 - m) * h
            logits = self.proj(h)
            outputs.append(logits)
            if random.random() < tf:
                token = tgt_emb[:, t, :]
            else:
                pred = logits.argmax(dim=-1)
                token = self.emb(pred)
        return torch.stack(
            outputs,
            dim=1
        )

encoder = EncoderGRU(d=64, src_vocab_size=len(src_vocab))
decoder = DecoderGRU(encoder, tgt_vocab_size=len(tgt_vocab))
decoder.to(device)
optimizer = torch.optim.Adam(
    decoder.parameters(),
    lr=0.001
)

In [ ]:
class EncoderBiGRU(nn.Module):
    def __init__(self, d, src_vocab_size):
        super().__init__()
        self.d = d
        self.emb = nn.Embedding(
            src_vocab_size,
            d,
            padding_idx=PAD_IDX
        )
        self.forward_cell = CustomGRUCell(
            d,
            d
        )
        self.backward_cell = CustomGRUCell(
            d,
            d
        )
        # Convert concatenated [hf, hb] -> decoder hidden size d
        self.hidden_bridge = nn.Linear(
            2 * d,
            d
        )
    def forward(
        self,
        x,
        mask,
        return_history=False
    ):
        x = self.emb(x)
        B, S, D = x.shape
        hf = torch.zeros(
            B,
            D,
            device=x.device,
            dtype=x.dtype
        )
        hb = torch.zeros(
            B,
            D,
            device=x.device,
            dtype=x.dtype
        )
        forward_history = []
        backward_history = [
            None
        ] * S
        for t in range(S):
            hf_new = self.forward_cell(
                x[:, t, :],
                hf
            )
            m = mask[:, t].unsqueeze(1).to(x.dtype)
            hf = m * hf_new + (1 - m) * hf
            forward_history.append(hf)
        for t in reversed(range(S)):
            hb_new = self.backward_cell(
                x[:, t, :],
                hb
            )
            m = mask[:, t].unsqueeze(1).to(x.dtype)
            hb = m * hb_new + (1 - m) * hb
            backward_history[t] = hb
        hf_history = torch.stack(
            forward_history,
            dim=1
        )
        hb_history = torch.stack(
            backward_history,
            dim=1
        )
        # (B, S, 2d)
        history = torch.cat(
            [hf_history, hb_history],
            dim=-1
        )
        # Last source representation
        # For padded sequences, using the final forward/backward
        # states as accumulated above is valid because padding
        # does not modify the recurrent state.
        final = torch.cat(
            [hf, hb],
            dim=-1
        )
        h = self.hidden_bridge(final)
        if return_history:
            return h, history
        return h

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(
        self,
        encoder_dim,
        decoder_dim,
        attention_dim
    ):
        super().__init__()
        self.W_h = nn.Linear(
            encoder_dim,
            attention_dim,
            bias=False
        )
        self.W_s = nn.Linear(
            decoder_dim,
            attention_dim,
            bias=False
        )
        self.v = nn.Linear(
            attention_dim,
            1,
            bias=False
        )

    def forward(
        self,
        decoder_hidden,
        encoder_outputs,
        src_mask
    ):
        """
        decoder_hidden:
            (B, d)

        encoder_outputs:
            (B, S, encoder_dim)

        src_mask:
            (B, S)

        returns:
            context:
                (B, encoder_dim)

            attention:
                (B, S)
        """
        # encoder:
        # (B,S,encoder_dim)
        enc_proj = self.W_h(
            encoder_outputs
        )
        # decoder:
        # (B,d) -> (B,1,attention_dim)
        dec_proj = self.W_s(
            decoder_hidden
        ).unsqueeze(1)
        energy = torch.tanh(
            enc_proj + dec_proj
        )
        # (B,S,1) -> (B,S)
        energy = self.v(
            energy
        ).squeeze(-1)
        # Mask padding
        energy = energy.masked_fill(
            src_mask == 0,
            -1e9
        )
        attention = torch.softmax(
            energy,
            dim=-1
        )
        # weighted encoder states
        context = torch.bmm(
            attention.unsqueeze(1),
            encoder_outputs
        ).squeeze(1)
        return context, attention

In [ ]:
class DecoderBiGRUAttention(nn.Module):
    def __init__(
        self,
        encoder,
        tgt_vocab_size,
        attention_dim=None
    ):
        super().__init__()
        self.encoder = encoder
        d = encoder.d
        if attention_dim is None:
            attention_dim = d
        self.emb = nn.Embedding(
            tgt_vocab_size,
            d,
            padding_idx=PAD_IDX
        )
        self.attention = BahdanauAttention(
            encoder_dim=2 * d,
            decoder_dim=d,
            attention_dim=attention_dim
        )
        # embedding d + context 2d
        self.cell = CustomGRUCell(
            input_size=3 * d,
            hidden_size=d
        )
        self.proj = nn.Linear(
            d,
            tgt_vocab_size,
            bias=False
        )
        self.proj.weight = self.emb.weight

    def forward(
        self,
        tgt,
        tgt_mask,
        src,
        src_mask,
        tf=0.5,
        return_attention=False
    ):
        tgt_emb = self.emb(tgt)
        B, T, D = tgt_emb.shape
        h, encoder_outputs = self.encoder(
            src,
            src_mask,
            return_history=True
        )
        # encoder_outputs:
        # (B,S,2d)
        token = tgt_emb[:, 0, :]
        outputs = []
        attention_history = []
        for t in range(1, T):
            # Bahdanau attention
            context, attention = self.attention(
                h,
                encoder_outputs,
                src_mask
            )
            # concatenate embedding + context
            decoder_input = torch.cat(
                [token, context],
                dim=-1
            )
            h_new = self.cell(
                decoder_input,
                h
            )
            m = tgt_mask[:, t].unsqueeze(1).to(
                tgt_emb.dtype
            )
            h = m * h_new + (1 - m) * h
            logits = self.proj(h)
            outputs.append(logits)
            if return_attention:
                attention_history.append(attention)
            # Teacher forcing
            if random.random() < tf:
                token = tgt_emb[:, t, :]
            else:
                pred = logits.argmax(
                    dim=-1
                )
                token = self.emb(pred)
        outputs = torch.stack(
            outputs,
            dim=1
        )
        if return_attention:
            attention_history = torch.stack(
                attention_history,
                dim=1
            )
            return outputs, attention_history
        return outputs
encoder = EncoderBiGRU(
    d=64,
    src_vocab_size=len(src_vocab)
)
decoder = DecoderBiGRUAttention(
    encoder=encoder,
    tgt_vocab_size=len(tgt_vocab),
    attention_dim=64
)
decoder.to(device)
optimizer = torch.optim.Adam(
    decoder.parameters(),
    lr=0.001
)

In [ ]:
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

@torch.no_grad()
def greedy_decode(model, src, src_mask, max_len=MAX_LEN):
    """
    Free-running (no teacher forcing) greedy decoding.
    model : a trained DecoderRNN (its .encoder is used to get h0)
    src, src_mask : (b, s) source ids / mask
    Returns: (b, max_len) tensor of predicted target ids (0 = <pad> after <eos>)
    """
    model.eval()
    b = src.shape[0]
    h = model.encoder(src, src_mask)                 # encoder final hidden state -> decoder h0
    token_id = torch.full((b,), SOS_IDX, dtype=torch.long, device=src.device)
    token = model.emb(token_id)                       # embedded <sos>
    finished = torch.zeros(b, dtype=torch.bool, device=src.device)
    out_ids = torch.full((b, max_len), PAD_IDX, dtype=torch.long, device=src.device)

    for t in range(1, max_len):
        h_1 = model.activation(token @ model.wxh + h @ model.whh + model.b)
        h = h_1  # no mask here: every position is "real" until a sequence finishes
        logits = model.proj(h)
        preds = logits.argmax(dim=1)
        # once a sequence has emitted <eos>, keep padding it instead of decoding further
        preds = torch.where(finished, torch.full_like(preds, PAD_IDX), preds)
        out_ids[:, t] = preds
        finished = finished | (preds == EOS_IDX)
        token = model.emb(preds)
        if finished.all():
            break
    return out_ids

BATCH_SIZE = 64

test_dataset = TranlationDataset(df_test, src_vocab, tgt_vocab)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

bucket_names = {0: "short", 1: "medium", 2: "long"}
hyps_by_bucket = {0: [], 1: [], 2: []}
refs_by_bucket = {0: [], 1: [], 2: []}

decoder.eval()
row = 0
with torch.no_grad():
    for src, tgt, label, src_mask, tgt_mask in tqdm(test_loader, desc="Greedy decoding test set"):
        src = src.to(device)
        src_mask = src_mask.to(device)

        pred_ids = greedy_decode(decoder, src, src_mask, max_len=MAX_LEN).cpu()
        bsz = src.shape[0]

        for i in range(bsz):
            hyp_tokens = tgt_vocab.decode(pred_ids[i].tolist()).split()
            ref_tokens = preprocess(df_test.iloc[row]["tgt"])  # raw ground-truth text, whitespace tokenized
            bucket = int(label[i].item())

            hyps_by_bucket[bucket].append(hyp_tokens)
            refs_by_bucket[bucket].append([ref_tokens])  # corpus_bleu expects a list of references per hypothesis
            row += 1

smoothie = SmoothingFunction().method4
bleu_scores = {}
for b in (0, 1, 2):
    name = bucket_names[b]
    if len(hyps_by_bucket[b]) == 0:
        bleu_scores[name] = float("nan")
        continue
    bleu_scores[name] = corpus_bleu(refs_by_bucket[b], hyps_by_bucket[b], smoothing_function=smoothie)

bleu_table = pd.DataFrame({
    "length_bucket": ["short", "medium", "long"],
    "n_examples": [len(hyps_by_bucket[0]), len(hyps_by_bucket[1]), len(hyps_by_bucket[2])],
    "corpus_BLEU": [bleu_scores["short"], bleu_scores["medium"], bleu_scores["long"]],
})
bleu_table

In [ ]:
qual_examples = [
    {
        "bucket": "Short",
        "src": "the old girl chases the red ball",
        "tgt": "redai balex oldai girol chaun",
    },
    {
        "bucket": "Medium",
        "src": "the old wizard throws the brave table and the blue fox finds the brave ball",
        "tgt": "braai balex bluai foxol finun yol braai tabex oldai wizol thrun",
    },
    {
        "bucket": "Long",
        "src": "the old dog finds the blue book and the swift cat sees the blue mountain and the small robot follows the old lamp and the young dog greets the quiet cloud",
        "tgt": "quiai cloex youai dogol greun yol oldai lamex smaai robol folun yol bluai mouex swiai catol seeun yol bluai booex oldai dogol finun",
    },
]

def encode_source(sentence, vocab, max_len=MAX_LEN):
    ids = [vocab.word2idx["<sos>"]] + vocab.encode(sentence)[:max_len - 2] + [vocab.word2idx["<eos>"]]
    mask = [1] * len(ids) + [0] * (max_len - len(ids))
    ids = ids + [PAD_IDX] * (max_len - len(ids))
    return (torch.tensor(ids, dtype=torch.long).unsqueeze(0),
            torch.tensor(mask, dtype=torch.long).unsqueeze(0))

decoder.eval()
with torch.no_grad():
    for ex in qual_examples:
        src_ids, src_mask = encode_source(ex["src"], src_vocab)
        src_ids, src_mask = src_ids.to(device), src_mask.to(device)

        pred_ids = greedy_decode(decoder, src_ids, src_mask, max_len=MAX_LEN)
        pred_text = tgt_vocab.decode(pred_ids[0].tolist())

        print(f"--- {ex['bucket']} ---")
        print(f"Source:       {ex['src']}")
        print(f"Ground Truth: {ex['tgt']}")
        print(f"Prediction:   {pred_text}")
        print()
eps = 1e-8
bleu_short = bleu_scores["short"]
bleu_medium = bleu_scores["medium"]
bleu_long = bleu_scores["long"]

delta_short_long = bleu_short - bleu_long
R_long = bleu_long / (bleu_short + eps)

print(f"BLEU_short  = {bleu_short:.4f}")
print(f"BLEU_medium = {bleu_medium:.4f}")
print(f"BLEU_long   = {bleu_long:.4f}")
print()
print(f"Delta_short-long = BLEU_short - BLEU_long = {delta_short_long:.4f}")
print(f"R_long           = BLEU_long / (BLEU_short + eps) = {R_long:.4f}")

if delta_short_long > 0:
    print("\nBLEU drops as sentences get longer: the fixed-size Seq2Seq bottleneck loses more "
          "information the more clauses it has to compress into a single hidden vector, so the "
          "decoder has a harder time reconstructing long-range structure.")

In [ ]:
@torch.no_grad()
def beam_decode(
    model,
    src,
    src_mask,
    max_len=MAX_LEN,
    num_beams=4,
):
    """
    Beam-search decoding for the custom RNN decoder.
    model : trained DecoderRNN
    src, src_mask : (b, s)
    max_len : maximum output length
    num_beams : number of beams

    Returns:
        out_ids : (b, max_len)
    """
    model.eval()
    b = src.size(0)
    device = src.device
    vocab_size = model.emb.num_embeddings
    h = model.encoder(src, src_mask)
    # Replicate encoder hidden state for every beam
    # (b, hidden) -> (b, beams, hidden) -> (b*beams, hidden)
    h = h.unsqueeze(1).expand(b, num_beams, -1)
    h = h.reshape(b * num_beams, -1)
    out_ids = torch.full(
        (b * num_beams, max_len),
        PAD_IDX,
        dtype=torch.long,
        device=device,
    )
    # Every beam starts with <SOS>
    out_ids[:, 0] = SOS_IDX
    beam_scores = torch.full(
        (b, num_beams),
        float("-inf"),
        device=device,
    )
    beam_scores[:, 0] = 0.0
    token_ids = torch.full(
        (b * num_beams,),
        SOS_IDX,
        dtype=torch.long,
        device=device,
    )
    finished = torch.zeros(
        b * num_beams,
        dtype=torch.bool,
        device=device,
    )
    for t in range(1, max_len):
        token = model.emb(token_ids)
        h_1 = model.activation(
            token @ model.wxh
            + h @ model.whh
            + model.b
        )
        logits = model.proj(h_1)
        log_probs = torch.log_softmax(logits, dim=-1)
        log_probs[finished] = float("-inf")
        log_probs[finished, PAD_IDX] = 0.0
        total_scores = (
            beam_scores.unsqueeze(-1)
            + log_probs.view(b, num_beams, vocab_size)
        )
        total_scores = total_scores.view(
            b,
            num_beams * vocab_size,
        )
        top_scores, top_indices = torch.topk(
            total_scores,
            k=num_beams,
            dim=-1,
        )
        next_beam = top_indices // vocab_size
        next_token = top_indices % vocab_size
        batch_offset = (
            torch.arange(b, device=device)
            .unsqueeze(1)
            * num_beams
        )
        flat_beam_indices = (
            batch_offset + next_beam
        ).reshape(-1)
        h = h_1[flat_beam_indices]
        old_finished = finished
        finished = old_finished[flat_beam_indices]
        finished = finished | (next_token.reshape(-1) == EOS_IDX)
        out_ids = out_ids[flat_beam_indices]
        out_ids[:, t] = next_token.reshape(-1)
        beam_scores = top_scores
        token_ids = next_token.reshape(-1)
        if finished.all():
            break
    best_beam = beam_scores.argmax(dim=1)
    batch_indices = torch.arange(
        b,
        device=device,
    )
    out_ids = out_ids.view(
        b,
        num_beams,
        max_len,
    )
    out_ids = out_ids[
        batch_indices,
        best_beam,
    ]
    return out_ids

In [ ]:
# ============================================================
# Logits processing
# ============================================================

def apply_repetition_penalty(
    logits: torch.Tensor,
    input_ids: torch.Tensor,
    penalty: float,
) -> torch.Tensor:
    """
    Repetition penalty from the token-generation literature / HF behavior.

    logits:    [B, V]
    input_ids: [B, T]
    """
    if penalty == 1.0:
        return logits

    if penalty <= 0:
        raise ValueError("repetition_penalty must be > 0.")

    logits = logits.clone()

    # Tokens appearing anywhere in the sequence are penalized once.
    for b in range(logits.size(0)):
        for token_id in torch.unique(input_ids[b]):
            token_id = token_id.item()
            value = logits[b, token_id]

            if value < 0:
                logits[b, token_id] = value * penalty
            else:
                logits[b, token_id] = value / penalty

    return logits


def apply_no_repeat_ngram(
    logits: torch.Tensor,
    input_ids: torch.Tensor,
    ngram_size: int,
) -> torch.Tensor:
    """
    Prevents generation of an n-gram that has already occurred.

    logits:    [B, V]
    input_ids: [B, T]
    """
    if ngram_size <= 0:
        return logits

    logits = logits.clone()

    if input_ids.size(1) < ngram_size - 1:
        return logits

    for b in range(input_ids.size(0)):
        seq = input_ids[b].tolist()

        if len(seq) < ngram_size - 1:
            continue

        # Map prefix -> tokens that followed that prefix previously.
        banned = {}

        for i in range(len(seq) - ngram_size + 1):
            prefix = tuple(seq[i : i + ngram_size - 1])
            next_token = seq[i + ngram_size - 1]

            banned.setdefault(prefix, set()).add(next_token)

        current_prefix = tuple(seq[-(ngram_size - 1):])

        for token_id in banned.get(current_prefix, set()):
            logits[b, token_id] = -float("inf")

    return logits


def apply_min_length(
    logits: torch.Tensor,
    input_ids: torch.Tensor,
    eos_token_id: Optional[int],
    min_length: int,
) -> torch.Tensor:
    """
    Prevent EOS before min_length tokens in the full sequence.
    """
    if eos_token_id is None:
        return logits

    if input_ids.size(1) < min_length:
        logits = logits.clone()
        logits[:, eos_token_id] = -float("inf")

    return logits


def apply_exponential_decay_length_penalty(
    logits: torch.Tensor,
    input_ids: torch.Tensor,
    eos_token_id: Optional[int],
    penalty: Optional[Tuple[int, float]],
) -> torch.Tensor:
    """
    Approximation of HF's exponential-decay length penalty.

    penalty = (start_index, decay_factor)

    After start_index, EOS becomes increasingly favored.
    """
    if penalty is None or eos_token_id is None:
        return logits

    start_index, decay_factor = penalty

    if decay_factor <= 0:
        raise ValueError("decay_factor must be > 0.")

    current_length = input_ids.size(1)

    if current_length < start_index:
        return logits

    logits = logits.clone()

    # Increasing EOS score according to sequence length.
    # This follows the intent of HF's processor:
    # make EOS progressively more likely after start_index.
    exponent = current_length - start_index + 1
    logits[:, eos_token_id] *= decay_factor ** exponent

    return logits


# ============================================================
# Sampling filters
# ============================================================

def top_k_filter(
    logits: torch.Tensor,
    top_k: int,
) -> torch.Tensor:
    if top_k <= 0:
        return logits

    vocab_size = logits.size(-1)
    top_k = min(top_k, vocab_size)

    values, _ = torch.topk(logits, top_k, dim=-1)
    threshold = values[:, -1].unsqueeze(-1)

    return torch.where(
        logits < threshold,
        torch.full_like(logits, -float("inf")),
        logits,
    )


def top_p_filter(
    logits: torch.Tensor,
    top_p: float,
) -> torch.Tensor:
    if top_p >= 1.0:
        return logits

    if not (0.0 < top_p <= 1.0):
        raise ValueError("top_p must be in (0, 1].")

    sorted_logits, sorted_indices = torch.sort(
        logits,
        descending=True,
        dim=-1,
    )

    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Remove tokens AFTER cumulative probability exceeds top_p.
    sorted_remove = cumulative_probs > top_p

    # Always keep the first token.
    sorted_remove[:, 0] = False

    # Shift mask so token causing threshold is retained.
    sorted_remove[:, 1:] = sorted_remove[:, :-1].clone()

    filtered_sorted = sorted_logits.masked_fill(
        sorted_remove,
        -float("inf"),
    )

    logits = torch.full_like(logits, -float("inf"))

    return logits.scatter(
        1,
        sorted_indices,
        filtered_sorted,
    )


def min_p_filter(
    logits: torch.Tensor,
    min_p: float,
) -> torch.Tensor:
    """
    Keep tokens whose probability is at least:

        min_p * max_probability
    """
    if min_p is None:
        return logits

    if not (0.0 <= min_p <= 1.0):
        raise ValueError("min_p must be in [0, 1].")

    if min_p == 0:
        return logits

    probs = F.softmax(logits, dim=-1)

    max_prob = probs.max(dim=-1, keepdim=True).values
    threshold = min_p * max_prob

    keep = probs >= threshold

    # Always keep the most likely token.
    top_token = probs.argmax(dim=-1, keepdim=True)
    keep.scatter_(1, top_token, True)

    return logits.masked_fill(~keep, -float("inf"))


def top_h_filter(
    logits: torch.Tensor,
    top_h: float,
) -> torch.Tensor:
    """
    Entropy-budget sampling.

    top_h = 1.0 -> keep the whole distribution
    smaller top_h -> more concentrated candidate set

    This follows the current HF definition: retain the smallest
    probability-sorted prefix whose renormalized entropy is <=
    top_h * original entropy.
    """
    if top_h is None or top_h >= 1.0:
        return logits

    if not (0.0 < top_h <= 1.0):
        raise ValueError("top_h must be in (0, 1].")

    batch_size, vocab_size = logits.shape

    probs = F.softmax(logits, dim=-1)

    sorted_probs, sorted_indices = torch.sort(
        probs,
        descending=True,
        dim=-1,
    )

    # Full entropy H(P)
    full_entropy = -torch.sum(
        probs * torch.log(probs.clamp_min(1e-20)),
        dim=-1,
        keepdim=True,
    )

    budget = top_h * full_entropy

    # Prefix cumulative probability.
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Prefix entropy after renormalization.
    # Compute for each prefix separately.
    prefix_entropy = torch.zeros_like(sorted_probs)

    for i in range(vocab_size):
        prefix = sorted_probs[:, : i + 1]
        p = prefix / prefix.sum(dim=-1, keepdim=True)

        entropy = -torch.sum(
            p * torch.log(p.clamp_min(1e-20)),
            dim=-1,
        )

        prefix_entropy[:, i] = entropy

    keep_count = torch.ones(
        batch_size,
        dtype=torch.long,
        device=logits.device,
    )

    for b in range(batch_size):
        valid = torch.nonzero(
            prefix_entropy[b] <= budget[b, 0],
            as_tuple=False,
        )

        if valid.numel() > 0:
            keep_count[b] = valid[-1, 0] + 1

    mask = torch.zeros_like(sorted_probs, dtype=torch.bool)

    for b in range(batch_size):
        mask[b, : keep_count[b]] = True

    filtered_sorted = torch.where(
        mask,
        sorted_probs,
        torch.zeros_like(sorted_probs),
    )

    # Convert probabilities back into log probabilities.
    filtered_sorted = torch.log(filtered_sorted.clamp_min(1e-20))

    output = torch.full_like(logits, -float("inf"))

    output.scatter_(
        1,
        sorted_indices,
        filtered_sorted,
    )

    return output


def process_logits(
    logits: torch.Tensor,
    input_ids: torch.Tensor,
    *,
    min_length: int,
    eos_token_id: Optional[int],
    temperature: float,
    top_k: int,
    top_p: float,
    min_p: Optional[float],
    top_h: Optional[float],
    repetition_penalty: float,
    no_repeat_ngram_size: int,
    exponential_decay_length_penalty: Optional[Tuple[int, float]],
) -> torch.Tensor:

    # Sequence-level processors
    logits = apply_repetition_penalty(
        logits,
        input_ids,
        repetition_penalty,
    )

    logits = apply_no_repeat_ngram(
        logits,
        input_ids,
        no_repeat_ngram_size,
    )

    logits = apply_min_length(
        logits,
        input_ids,
        eos_token_id,
        min_length,
    )

    logits = apply_exponential_decay_length_penalty(
        logits,
        input_ids,
        eos_token_id,
        exponential_decay_length_penalty,
    )

    # Temperature
    if temperature <= 0:
        raise ValueError("temperature must be > 0.")

    logits = logits / temperature

    # Sampling filters
    logits = top_k_filter(logits, top_k)
    logits = top_p_filter(logits, top_p)
    logits = min_p_filter(logits, min_p)
    logits = top_h_filter(logits, top_h)

    return logits


# ============================================================
# Token selection
# ============================================================

def select_token(
    logits: torch.Tensor,
    do_sample: bool,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Returns:

        next_token: [B, 1]
        next_score: [B]
    """
    log_probs = F.log_softmax(logits, dim=-1)

    if do_sample:
        probs = torch.exp(log_probs)

        next_token = torch.multinomial(
            probs,
            num_samples=1,
        )

        next_score = torch.gather(
            log_probs,
            1,
            next_token,
        ).squeeze(1)

    else:
        next_score, next_token = torch.max(
            log_probs,
            dim=-1,
            keepdim=True,
        )

        next_token = next_token
        next_score = next_score.squeeze(1)

    return next_token, next_score